### Coleta de dados de Mortalidade

https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/csv/Mortalidade_Geral_2025_csv.zip


In [ ]:
import sys, os
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

In [ ]:
# https://servicodados.ibge.gov.br/api/v1/localidades/estados

ufs = {"11":"RO", "12":"AC", "13":"AM", "14":"RR", "15":"PA", "16":"AP", "17":"TO"
      ,"21":"MA", "22":"PI", "23":"CE", "24":"RN", "25":"PB", "26":"PE", "27":"AL"
      ,"28":"SE", "29":"BA", "31":"MG", "32":"ES", "33":"RJ", "35":"SP"
      ,"41":"PR", "42":"SC", "43":"RS", "50":"MS", "51":"MT", "52":"GO", "53":"DF"
      }


In [ ]:
df_dengue = \
    spark.read.csv(f"{PROJECT_PATH}\\DENGBR26.CSV"
                  ,header=True
                  ,sep=","
                  ,inferSchema=True)

In [ ]:
df_dengue.printSchema()

In [ ]:
df_dengue.select("UF", "coufinf").drop_duplicates().show(1000,False)

In [ ]:
expr = F.create_map([F.lit(x) for kv in ufs.items() for x in kv])

df_dengue = \
    df_dengue.withColumns({"MES": F.month(F.col('DT_NOTIFIC'))
                          ,"sigla_uf": expr[F.col("sg_uf_not")] })

df_dengue_cols = \
    (df_dengue
        .select("TP_NOT"
            ,"NU_ANO"
            ,"MES"
            ,F.col("NU_IDADE_N").substr(2,3).cast("int").alias("IDADE")
            ,"ID_MUNICIP"
            ,"sg_uf_not"
            ,"sigla_uf"
            ,"sg_uf"))

df_dengue_cols.createOrReplaceTempView("notif_dengue")

In [ ]:
query_dengue = \
    """Select sg_uf, nu_ano, count(1)
         from notif_dengue
        where 1=1
          -- id_municip in (110001, 110003, 110002) 
        group by all 
    """

spark.sql(query_dengue).orderBy('sig_uf').show()

In [9]:
colunas = ["contador","ORIGEM","TIPOBITO","DTOBITO","HORAOBITO","NATURAL","CODMUNNATU","DTNASC","IDADE","SEXO","RACACOR","ESTCIV","ESC","ESC2010","SERIESCFAL","OCUP","CODMUNRES","LOCOCOR","CODESTAB","CODMUNOCOR","IDADEMAE","ESCMAE","ESCMAE2010","SERIESCMAE","OCUPMAE","QTDFILVIVO","QTDFILMORT","GRAVIDEZ","SEMAGESTAC","GESTACAO","PARTO","OBITOPARTO","PESO","TPMORTEOCO","OBITOGRAV","OBITOPUERP","ASSISTMED","EXAME","CIRURGIA","NECROPSIA","LINHAA","LINHAB","LINHAC","LINHAD","LINHAII","CAUSABAS","CB_PRE","COMUNSVOIM","DTATESTADO","CIRCOBITO","ACIDTRAB","FONTE","NUMEROLOTE","DTINVESTIG","DTCADASTRO","ATESTANTE","STCODIFICA","CODIFICADO","VERSAOSIST","VERSAOSCB","FONTEINV","DTRECEBIM","ATESTADO","DTRECORIGA","OPOR_DO","CAUSAMAT","ESCMAEAGR1","ESCFALAGR1","STDOEPIDEM","STDONOVA","DIFDATA","NUDIASOBCO","DTCADINV","TPOBITOCOR","DTCONINV","FONTES","TPRESGINFO","TPNIVELINV","DTCADINF","MORTEPARTO","DTCONCASO","ALTCAUSA","CAUSABAS_O","TPPOS","TP_ALTERA","CB_ALT","MAT_CLAS","COVID_CLAS"]

for col in colunas:
    print(col)

contador
ORIGEM
TIPOBITO
DTOBITO
HORAOBITO
NATURAL
CODMUNNATU
DTNASC
IDADE
SEXO
RACACOR
ESTCIV
ESC
ESC2010
SERIESCFAL
OCUP
CODMUNRES
LOCOCOR
CODESTAB
CODMUNOCOR
IDADEMAE
ESCMAE
ESCMAE2010
SERIESCMAE
OCUPMAE
QTDFILVIVO
QTDFILMORT
GRAVIDEZ
SEMAGESTAC
GESTACAO
PARTO
OBITOPARTO
PESO
TPMORTEOCO
OBITOGRAV
OBITOPUERP
ASSISTMED
EXAME
CIRURGIA
NECROPSIA
LINHAA
LINHAB
LINHAC
LINHAD
LINHAII
CAUSABAS
CB_PRE
COMUNSVOIM
DTATESTADO
CIRCOBITO
ACIDTRAB
FONTE
NUMEROLOTE
DTINVESTIG
DTCADASTRO
ATESTANTE
STCODIFICA
CODIFICADO
VERSAOSIST
VERSAOSCB
FONTEINV
DTRECEBIM
ATESTADO
DTRECORIGA
OPOR_DO
CAUSAMAT
ESCMAEAGR1
ESCFALAGR1
STDOEPIDEM
STDONOVA
DIFDATA
NUDIASOBCO
DTCADINV
TPOBITOCOR
DTCONINV
FONTES
TPRESGINFO
TPNIVELINV
DTCADINF
MORTEPARTO
DTCONCASO
ALTCAUSA
CAUSABAS_O
TPPOS
TP_ALTERA
CB_ALT
MAT_CLAS
COVID_CLAS


In [6]:
# path_contrato = r"C:\Users\DRT90628\Downloads\urn-datacontract-edgeconsumerbigdata-hive-raw_datasus-raw_tb_datasus_sihsus_rd.yaml"
path_contrato = r"C:\Users\DRT90628\Downloads\urn-datacontract-edgeconsumerbigdata-hive-raw_datasus-raw_tb_datasus_sim.yaml"

# id_contrato: urn:datacontract:edgeconsumerbigdata:hive:raw_datasus:datasus_notificacao_casos_dengue

In [8]:
import yaml

# 1. Carrega o arquivo YAML
with open(path_contrato, "r", encoding="utf-8") as file:
    data = yaml.safe_load(file)

# 2. Navega até o dicionário de campos do modelo especificado
try:
    fields = data["models"]["raw_tb_datasus_sim"]["fields"]
    
    # 3. Itera sobre os campos (chave: nome da coluna, valor: atributos da coluna)
    for field_name, field_info in fields.items():
        field_type = field_info.get("type")
        description = field_info.get("description")
        required = field_info.get("required")
        primary_key = field_info.get("primaryKey")
        precision = field_info.get("precision")


        print(f"{field_name};{field_type};{precision}")

        
        # # Exemplo de saída/processamento com os dados obtidos
        # print(f"Campo: {field_name}")
        # print(f"  - Tipo: {field_type}")
        # print(f"  - Obrigatório: {required}")
        # print(f"  - Chave Primária: {primary_key}")
        # print(f"  - Descrição: {description}")
        # print("-" * 50)

except KeyError as e:
    print(f"Erro ao acessar a chave no YAML: {e}")

codmunocor;varchar;None
fonte;varchar;None
linhac;varchar;None
qtdfilvivo;varchar;None
acidtrab;varchar;None
contador;varchar;None
assistmed;varchar;None
exame;varchar;None
parto;varchar;None
estciv;varchar;None
dtobito;varchar;None
tipobito;varchar;None
idade;varchar;None
linhaii;varchar;None
ocupmae;varchar;None
codmunres;varchar;None
dtnasc;varchar;None
gestacao;varchar;None
linhab;varchar;None
natural;varchar;None
esc;varchar;None
qtdfilmort;varchar;None
obitopuerp;varchar;None
idademae;varchar;None
cirurgia;varchar;None
linhad;varchar;None
peso;varchar;None
obitograv;varchar;None
racacor;varchar;None
codbaires;varchar;None
escmae;varchar;None
obitoparto;varchar;None
causabas;varchar;None
lococor;varchar;None
gravidez;varchar;None
linhaa;varchar;None
circobito;varchar;None
sexo;varchar;None
necropsia;varchar;None
ocup;varchar;None
